
# 15. Official PatchTST H=96 Baseline Reproduction

## 목적

이번 실험은 retrieval을 붙이기 전에 **강한 PatchTST baseline을 먼저 확정**하기 위한 실험입니다.

대상:

- ETTh1
- ETTm1
- prediction length \(H=96\)

공식 PatchTST supervised repository의 **실제 `PatchTST_supervised/models/PatchTST.py`와 backbone**을 사용합니다.

---

# 왜 별도로 재현하는가?

이전 screening에서는 Time-Series-Library의 PatchTST reimplementation을 사용했고:

- `seq_len=96`
- 동일한 128/8-head architecture
- AdamW + cosine
- batch 32
- max 30 epochs

를 모든 데이터셋에 적용했습니다.

하지만 공식 PatchTST supervised scripts는 다음처럼 dataset-specific입니다.

### ETTh1 official recipe

\[
L=336,\quad H=96
\]

- `enc_in=7`
- `e_layers=3`
- `n_heads=4`
- `d_model=16`
- `d_ff=128`
- `dropout=0.3`
- `fc_dropout=0.3`
- `head_dropout=0`
- `patch_len=16`
- `stride=8`
- batch 128
- 100 epochs
- learning rate \(10^{-4}\)
- seed 2021
- default `patience=100`
- default `lradj=type3`
- default `pct_start=0.3`
- RevIN on, affine off
- replication padding at end

### ETTm1 official recipe

\[
L=336,\quad H=96
\]

- `enc_in=7`
- `e_layers=3`
- `n_heads=16`
- `d_model=128`
- `d_ff=256`
- `dropout=0.2`
- `fc_dropout=0.2`
- `head_dropout=0`
- `patch_len=16`
- `stride=8`
- batch 128
- 100 epochs
- patience 20
- learning rate \(10^{-4}\)
- `lradj=TST`
- `pct_start=0.4`
- seed 2021
- RevIN on, affine off
- replication padding at end

---

# 이번 notebook의 fidelity level

공식 repository의 model/backbone을 그대로 import합니다.

다음도 공식 code와 맞춥니다.

- train-only StandardScaler
- ETT train/val/test borders
- shuffled train/validation loader
- Adam optimizer
- MSE loss
- validation early stopping
- ETTh1 `type3` LR adjustment
- ETTm1 OneCycleLR / `TST`
- seed 2021
- official dataset-specific model hyperparameters

단 한 가지 의도적 차이:

> 공식 training loop는 매 epoch test loss도 출력하지만, test loss를 early stopping이나 optimizer에 사용하지 않습니다.

이번 notebook에서는 **test를 training 중에 보지 않습니다.**
이는 model update와 validation-based model selection을 바꾸지 않으면서 protocol을 더 엄격하게 유지하기 위한 것입니다.

---

# 두 종류의 test metric을 모두 출력

## 1. `OfficialStyle`

공식 `data_factory.py`와 동일하게:

- batch size 128
- `drop_last=True`

따라서 마지막 incomplete batch는 제외됩니다.

## 2. `FullStride1`

우리 retrieval paper protocol에 맞게:

- 모든 valid test origin
- stride 1
- drop 없음

최종 retrieval augmentation과 비교할 때는 **반드시 `FullStride1`**을 사용합니다.

이렇게 하면

\[
\boxed{\text{official reproduction}}
\]

과

\[
\boxed{\text{matched retrieval evaluation}}
\]

을 동시에 확보할 수 있습니다.


In [1]:

from pathlib import Path
from contextlib import nullcontext
from dataclasses import dataclass, asdict
from types import SimpleNamespace
import gc
import importlib
import json
import os
import random
import shutil
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 360)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = False  # official script default: False

DATASETS = ["ETTh1", "ETTm1"]
SEQ_LEN = 336
PRED_LEN = 96
LABEL_LEN = 48

SEED = 2021

RESULT_ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "official_patchtst_h96_reproduction"
)
CKPT_ROOT = RESULT_ROOT / "checkpoints"
HISTORY_ROOT = RESULT_ROOT / "histories"
ARTIFACT_ROOT = RESULT_ROOT / "artifacts"

for p in [RESULT_ROOT, CKPT_ROOT, HISTORY_ROOT, ARTIFACT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

RESUME = True
FORCE_RETRAIN = False

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("Output:", RESULT_ROOT)


Device: cuda
PyTorch: 2.4.1+cu121
Output: /data/dataset/strong_forecaster/official_patchtst_h96_reproduction


## 1. Locate or clone the official PatchTST repository

In [2]:

# Official repository:
# https://github.com/yuqinie98/PatchTST

OFFICIAL_REPO_CANDIDATES = [
    Path(
        "/code/stock_regime_retrieval/"
        "strong_forecaster/PatchTST_official"
    ),
    Path("/code/PatchTST_official"),
    Path("/data/PatchTST_official"),
    Path("/data/PatchTST"),
]

OFFICIAL_REPO = next(
    (
        p for p in OFFICIAL_REPO_CANDIDATES
        if (
            p
            / "PatchTST_supervised"
            / "models"
            / "PatchTST.py"
        ).exists()
    ),
    None,
)

if OFFICIAL_REPO is None:
    OFFICIAL_REPO = OFFICIAL_REPO_CANDIDATES[0]
    OFFICIAL_REPO.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(
        "Official repository not found locally. "
        "Cloning into:",
        OFFICIAL_REPO,
    )

    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/yuqinie98/PatchTST.git",
            str(OFFICIAL_REPO),
        ],
        check=True,
    )

SUPERVISED_ROOT = (
    OFFICIAL_REPO
    / "PatchTST_supervised"
)

if not (
    SUPERVISED_ROOT
    / "models"
    / "PatchTST.py"
).exists():
    raise FileNotFoundError(
        f"Official supervised code not found: {SUPERVISED_ROOT}"
    )

try:
    commit = subprocess.check_output(
        [
            "git",
            "-C",
            str(OFFICIAL_REPO),
            "rev-parse",
            "HEAD",
        ],
        text=True,
    ).strip()
except Exception:
    commit = "unknown"

print("Official repo:", OFFICIAL_REPO)
print("Supervised root:", SUPERVISED_ROOT)
print("Git commit:", commit)

(ARTIFACT_ROOT / "official_repo_commit.txt").write_text(
    commit + "\n"
)


Official repository not found locally. Cloning into: /code/stock_regime_retrieval/strong_forecaster/PatchTST_official


Cloning into '/code/stock_regime_retrieval/strong_forecaster/PatchTST_official'...


Official repo: /code/stock_regime_retrieval/strong_forecaster/PatchTST_official
Supervised root: /code/stock_regime_retrieval/strong_forecaster/PatchTST_official/PatchTST_supervised
Git commit: 204c21efe0b39603ad6e2ca640ef5896646ab1a9


41

## 2. Import the official PatchTST model without mixing it with TSLib

In [3]:

# The user's environment also contains Time-Series-Library, which has
# modules named `models` and `layers`. Remove already imported copies
# so Python cannot accidentally load the wrong implementation.

for module_name in list(sys.modules.keys()):
    if (
        module_name == "models"
        or module_name.startswith("models.")
        or module_name == "layers"
        or module_name.startswith("layers.")
    ):
        del sys.modules[module_name]

if str(SUPERVISED_ROOT) in sys.path:
    sys.path.remove(
        str(SUPERVISED_ROOT)
    )

sys.path.insert(
    0,
    str(SUPERVISED_ROOT),
)

OfficialPatchTST = importlib.import_module(
    "models.PatchTST"
).Model

model_file = Path(
    importlib.import_module(
        "models.PatchTST"
    ).__file__
).resolve()

print("Imported PatchTST from:", model_file)

expected = (
    SUPERVISED_ROOT
    / "models"
    / "PatchTST.py"
).resolve()

if model_file != expected:
    raise RuntimeError(
        "Wrong PatchTST implementation imported!\n"
        f"Expected: {expected}\n"
        f"Actual:   {model_file}"
    )

print("PASS: official PatchTST implementation is active.")


Imported PatchTST from: /code/stock_regime_retrieval/strong_forecaster/PatchTST_official/PatchTST_supervised/models/PatchTST.py
PASS: official PatchTST implementation is active.


## 3. Dataset paths and standard ETT split

In [4]:

DATA_PATHS = {
    "ETTh1": next(
        (
            p for p in [
                Path(
                    "/data/Time-Series-Library/"
                    "dataset/ETT-small/ETTh1.csv"
                ),
                Path("/data/dataset/ETTh1.csv"),
            ]
            if p.is_file()
        ),
        None,
    ),
    "ETTm1": Path(
        "/data/dataset/ETTm1.csv"
    ),
}

for name, p in DATA_PATHS.items():
    if p is None or not p.is_file():
        raise FileNotFoundError(
            f"{name} dataset not found: {p}"
        )

    print(
        f"{name:6s} -> {p}"
    )


def ett_unit(name):
    if name == "ETTh1":
        return 30 * 24

    if name == "ETTm1":
        return 30 * 24 * 4

    raise ValueError(name)


def split_bounds(name):
    u = ett_unit(name)

    train_end = 12 * u
    val_end = 16 * u
    test_end = 20 * u

    return train_end, val_end, test_end


def load_raw_values(name):
    path = DATA_PATHS[name]

    df = pd.read_csv(path)

    if "date" not in df.columns:
        raise ValueError(
            f"{name}: official ETT CSV requires a date column. "
            f"Columns={df.columns.tolist()[:20]}"
        )

    value_cols = [
        c for c in df.columns
        if c != "date"
    ]

    raw = df[
        value_cols
    ].to_numpy(
        dtype=np.float32
    )

    tr, va, te = split_bounds(name)

    if len(raw) < te:
        raise ValueError(
            f"{name}: expected at least {te} rows, got {len(raw)}."
        )

    raw = raw[:te]

    if raw.shape[1] != 7:
        raise ValueError(
            f"{name}: official script uses enc_in=7, "
            f"but CSV has {raw.shape[1]} numeric series."
        )

    return {
        "df": df,
        "columns": value_cols,
        "raw": raw,
        "train_end": tr,
        "val_end": va,
        "test_end": te,
    }


DATA = {
    name: load_raw_values(name)
    for name in DATASETS
}

display(
    pd.DataFrame([
        {
            "Dataset": name,
            "Rows": d["test_end"],
            "Channels": len(d["columns"]),
            "TrainEnd": d["train_end"],
            "ValEnd": d["val_end"],
            "TestEnd": d["test_end"],
            "Columns": ",".join(d["columns"]),
        }
        for name, d in DATA.items()
    ])
)


ETTh1  -> /data/Time-Series-Library/dataset/ETT-small/ETTh1.csv
ETTm1  -> /data/dataset/ETTm1.csv


,Dataset,Rows,Channels,TrainEnd,ValEnd,TestEnd,Columns
0,ETTh1,14400,7,8640,11520,14400,"HUFL,HULL,MUFL,MULL,LUFL,LULL,OT"
1,ETTm1,57600,7,34560,46080,57600,"HUFL,HULL,MUFL,MULL,LUFL,LULL,OT"


## 4. Faithful ETT Dataset implementation

In [5]:

class ETTForecastDataset(Dataset):
    def __init__(
        self,
        name,
        flag,
        seq_len=SEQ_LEN,
        label_len=LABEL_LEN,
        pred_len=PRED_LEN,
    ):
        super().__init__()

        assert flag in {
            "train",
            "val",
            "test",
        }

        self.name = name
        self.flag = flag
        self.seq_len = seq_len
        self.label_len = label_len
        self.pred_len = pred_len

        d = DATA[name]
        raw = d["raw"]

        # Official Dataset_ETT_hour/minute:
        # scaler is fit on the first 12 months only.
        self.scaler = StandardScaler()

        self.scaler.fit(
            raw[
                :d["train_end"]
            ]
        )

        scaled = self.scaler.transform(
            raw
        ).astype(
            np.float32
        )

        border1s = [
            0,
            d["train_end"] - seq_len,
            d["val_end"] - seq_len,
        ]

        border2s = [
            d["train_end"],
            d["val_end"],
            d["test_end"],
        ]

        type_map = {
            "train": 0,
            "val": 1,
            "test": 2,
        }

        idx = type_map[
            flag
        ]

        self.border1 = border1s[
            idx
        ]

        self.border2 = border2s[
            idx
        ]

        self.data_x = scaled[
            self.border1:
            self.border2
        ]

        self.data_y = self.data_x

    def __getitem__(
        self,
        index,
    ):
        s_begin = index
        s_end = (
            s_begin
            + self.seq_len
        )

        r_begin = (
            s_end
            - self.label_len
        )

        r_end = (
            r_begin
            + self.label_len
            + self.pred_len
        )

        seq_x = self.data_x[
            s_begin:s_end
        ]

        seq_y = self.data_y[
            r_begin:r_end
        ]

        return (
            torch.from_numpy(
                seq_x
            ),
            torch.from_numpy(
                seq_y
            ),
        )

    def __len__(
        self,
    ):
        return (
            len(
                self.data_x
            )
            - self.seq_len
            - self.pred_len
            + 1
        )


for name in DATASETS:
    for flag in [
        "train",
        "val",
        "test",
    ]:
        ds = ETTForecastDataset(
            name,
            flag,
        )

        print(
            f"{name:6s} {flag:5s}: "
            f"{len(ds):,} windows"
        )


ETTh1  train: 8,209 windows
ETTh1  val  : 2,785 windows
ETTh1  test : 2,785 windows
ETTm1  train: 34,129 windows
ETTm1  val  : 11,425 windows
ETTm1  test : 11,425 windows


## 5. Exact official H=96 recipes

In [6]:

OFFICIAL_RECIPES = {
    "ETTh1": {
        "seq_len": 336,
        "pred_len": 96,
        "enc_in": 7,
        "e_layers": 3,
        "n_heads": 4,
        "d_model": 16,
        "d_ff": 128,
        "dropout": 0.3,
        "fc_dropout": 0.3,
        "head_dropout": 0.0,
        "patch_len": 16,
        "stride": 8,
        "batch_size": 128,
        "train_epochs": 100,
        "patience": 100,
        "learning_rate": 1e-4,
        "lradj": "type3",
        "pct_start": 0.3,
        "random_seed": 2021,
    },
    "ETTm1": {
        "seq_len": 336,
        "pred_len": 96,
        "enc_in": 7,
        "e_layers": 3,
        "n_heads": 16,
        "d_model": 128,
        "d_ff": 256,
        "dropout": 0.2,
        "fc_dropout": 0.2,
        "head_dropout": 0.0,
        "patch_len": 16,
        "stride": 8,
        "batch_size": 128,
        "train_epochs": 100,
        "patience": 20,
        "learning_rate": 1e-4,
        "lradj": "TST",
        "pct_start": 0.4,
        "random_seed": 2021,
    },
}

display(
    pd.DataFrame(
        OFFICIAL_RECIPES
    ).T
)


,seq_len,pred_len,enc_in,e_layers,n_heads,d_model,d_ff,dropout,fc_dropout,head_dropout,patch_len,stride,batch_size,train_epochs,patience,learning_rate,lradj,pct_start,random_seed
ETTh1,336,96,7,3,4,16,128,0.3,0.3,0.0,16,8,128,100,100,0.0001,type3,0.3,2021
ETTm1,336,96,7,3,16,128,256,0.2,0.2,0.0,16,8,128,100,20,0.0001,TST,0.4,2021


## 6. Build the official PatchTST model

In [7]:

def make_official_config(
    name,
):
    r = OFFICIAL_RECIPES[
        name
    ]

    return SimpleNamespace(
        enc_in=
            r[
                "enc_in"
            ],
        seq_len=
            r[
                "seq_len"
            ],
        pred_len=
            r[
                "pred_len"
            ],
        e_layers=
            r[
                "e_layers"
            ],
        n_heads=
            r[
                "n_heads"
            ],
        d_model=
            r[
                "d_model"
            ],
        d_ff=
            r[
                "d_ff"
            ],
        dropout=
            r[
                "dropout"
            ],
        fc_dropout=
            r[
                "fc_dropout"
            ],
        head_dropout=
            r[
                "head_dropout"
            ],
        individual=0,
        patch_len=
            r[
                "patch_len"
            ],
        stride=
            r[
                "stride"
            ],
        padding_patch="end",
        revin=1,
        affine=0,
        subtract_last=0,
        decomposition=0,
        kernel_size=25,
    )


def build_official_model(
    name,
):
    cfg = make_official_config(
        name
    )

    model = OfficialPatchTST(
        cfg
    ).float().to(
        DEVICE
    )

    return model


for name in DATASETS:
    m = build_official_model(
        name
    )

    params = sum(
        p.numel()
        for p in m.parameters()
        if p.requires_grad
    )

    print(
        f"{name:6s}: "
        f"{params/1e6:.3f} M trainable params"
    )

    del m

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


ETTh1 : 0.082 M trainable params
ETTm1 : 0.921 M trainable params


## 7. Seed, loaders, and official optimizer/LR behavior

In [8]:

def set_seed(
    seed,
):
    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )

    # Do not force deterministic algorithms that were not used
    # by the original script.
    torch.backends.cudnn.benchmark = False


def make_loaders(
    name,
):
    r = OFFICIAL_RECIPES[
        name
    ]

    train_ds = ETTForecastDataset(
        name,
        "train",
    )

    val_ds = ETTForecastDataset(
        name,
        "val",
    )

    test_ds = ETTForecastDataset(
        name,
        "test",
    )

    # Official data_factory:
    # train and val: shuffle=True, drop_last=True
    # test: shuffle=False, drop_last=True
    train_loader = DataLoader(
        train_ds,
        batch_size=
            r[
                "batch_size"
            ],
        shuffle=True,
        num_workers=0,
        drop_last=True,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=
            r[
                "batch_size"
            ],
        shuffle=True,
        num_workers=0,
        drop_last=True,
    )

    official_test_loader = DataLoader(
        test_ds,
        batch_size=
            r[
                "batch_size"
            ],
        shuffle=False,
        num_workers=0,
        drop_last=True,
    )

    # Matched evaluation for our retrieval protocol:
    # all stride-1 windows.
    full_test_loader = DataLoader(
        test_ds,
        batch_size=
            r[
                "batch_size"
            ],
        shuffle=False,
        num_workers=0,
        drop_last=False,
    )

    return {
        "train_ds":
            train_ds,
        "val_ds":
            val_ds,
        "test_ds":
            test_ds,
        "train":
            train_loader,
        "val":
            val_loader,
        "official_test":
            official_test_loader,
        "full_test":
            full_test_loader,
    }


def adjust_type3_lr(
    optimizer,
    base_lr,
    epoch,
):
    # Official utils/tools.py:
    # lr = base_lr for epoch < 3,
    # else base_lr * 0.9 ** (epoch - 3)
    lr = (
        base_lr
        if epoch < 3
        else
        base_lr
        * (
            0.9
            ** (
                epoch-3
            )
        )
    )

    for group in optimizer.param_groups:
        group[
            "lr"
        ] = lr

    return lr


## 8. Validation and test metrics

In [9]:

@torch.no_grad()
def evaluate_loader(
    model,
    loader,
):
    model.eval()

    sse = 0.0
    sae = 0.0
    count = 0

    batch_mse = []

    for batch_x, batch_y in loader:
        x = batch_x.float().to(
            DEVICE
        )

        y = batch_y[
            :,
            -PRED_LEN:,
            :
        ].float().to(
            DEVICE
        )

        pred = model(
            x
        )

        pred = pred[
            :,
            -PRED_LEN:,
            :
        ]

        err = (
            pred-y
        )

        batch_mse.append(
            float(
                (
                    err**2
                ).mean().item()
            )
        )

        sse += float(
            (
                err**2
            ).sum().item()
        )

        sae += float(
            err.abs()
            .sum().item()
        )

        count += err.numel()

        del x, y, pred, err

    # Official vali() averages per-batch MSE.
    official_average_batch_mse = float(
        np.mean(
            batch_mse
        )
    )

    # Elementwise global metric, equivalent when all batches
    # have equal size; needed for full-test drop_last=False.
    global_mse = sse/count
    global_mae = sae/count

    return {
        "BatchAverageMSE":
            official_average_batch_mse,
        "MSE":
            global_mse,
        "MAE":
            global_mae,
        "Elements":
            count,
        "Batches":
            len(
                batch_mse
            ),
    }


## 9. Official-recipe training loop

In [10]:

def checkpoint_path(
    name,
):
    return (
        CKPT_ROOT
        / f"{name}_L336_H96_official_recipe_seed2021.pt"
    )


def history_path(
    name,
):
    return (
        HISTORY_ROOT
        / f"{name}_L336_H96_official_recipe_seed2021.csv"
    )


def train_or_load(
    name,
):
    r = OFFICIAL_RECIPES[
        name
    ]

    ckpt_path = checkpoint_path(
        name
    )

    if (
        RESUME
        and ckpt_path.exists()
        and not FORCE_RETRAIN
    ):
        ckpt = torch.load(
            ckpt_path,
            map_location=DEVICE,
        )

        model = build_official_model(
            name
        )

        model.load_state_dict(
            ckpt[
                "StateDict"
            ]
        )

        model.eval()

        print(
            f"Loaded {name}: "
            f"best val={ckpt['BestValMSE']:.6f} "
            f"epoch={ckpt['BestEpoch']}"
        )

        return (
            model,
            ckpt,
            make_loaders(
                name
            ),
        )

    set_seed(
        r[
            "random_seed"
        ]
    )

    loaders = make_loaders(
        name
    )

    model = build_official_model(
        name
    )

    # Official exp_main.py uses optim.Adam, not AdamW.
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=
            r[
                "learning_rate"
            ],
    )

    train_steps = len(
        loaders[
            "train"
        ]
    )

    onecycle = torch.optim.lr_scheduler.OneCycleLR(
        optimizer=
            optimizer,
        steps_per_epoch=
            train_steps,
        pct_start=
            r[
                "pct_start"
            ],
        epochs=
            r[
                "train_epochs"
            ],
        max_lr=
            r[
                "learning_rate"
            ],
    )

    criterion = nn.MSELoss()

    best_val = float(
        "inf"
    )

    best_epoch = -1
    best_state = None
    wait = 0
    history = []

    for epoch_idx in range(
        r[
            "train_epochs"
        ]
    ):
        epoch = (
            epoch_idx+1
        )

        model.train()

        losses = []

        epoch_start = time.time()

        for batch_x, batch_y in loaders[
            "train"
        ]:
            x = batch_x.float().to(
                DEVICE
            )

            y = batch_y[
                :,
                -PRED_LEN:,
                :
            ].float().to(
                DEVICE
            )

            optimizer.zero_grad()

            pred = model(
                x
            )

            pred = pred[
                :,
                -PRED_LEN:,
                :
            ]

            loss = criterion(
                pred,
                y,
            )

            loss.backward()

            optimizer.step()

            losses.append(
                float(
                    loss.item()
                )
            )

            # Official code steps OneCycleLR per batch only
            # when lradj == 'TST'.
            if r[
                "lradj"
            ] == "TST":
                # Official order calls adjust_learning_rate(...)
                # with scheduler.get_last_lr(), then scheduler.step().
                current = onecycle.get_last_lr()[
                    0
                ]

                for group in optimizer.param_groups:
                    group[
                        "lr"
                    ] = current

                onecycle.step()

            del (
                x,
                y,
                pred,
                loss,
            )

        train_loss = float(
            np.mean(
                losses
            )
        )

        val = evaluate_loader(
            model,
            loaders[
                "val"
            ],
        )

        # Official EarlyStopping uses validation loss only.
        # Its validation loss is the average of batch MSE.
        val_for_selection = val[
            "BatchAverageMSE"
        ]

        if (
            val_for_selection
            < best_val
        ):
            best_val = val_for_selection
            best_epoch = epoch

            best_state = {
                k:
                    v.detach()
                    .cpu()
                    .clone()
                for k, v
                in model.state_dict().items()
            }

            wait = 0

        else:
            wait += 1

        # Official type3 adjustment happens once after each epoch.
        if r[
            "lradj"
        ] != "TST":
            lr = adjust_type3_lr(
                optimizer,
                r[
                    "learning_rate"
                ],
                epoch,
            )
        else:
            lr = onecycle.get_last_lr()[
                0
            ]

        history.append({
            "Epoch":
                epoch,
            "TrainMSE":
                train_loss,
            "ValBatchAverageMSE":
                val[
                    "BatchAverageMSE"
                ],
            "ValGlobalMSE":
                val[
                    "MSE"
                ],
            "ValMAE":
                val[
                    "MAE"
                ],
            "LR":
                lr,
            "BestEpochSoFar":
                best_epoch,
            "BestValSoFar":
                best_val,
            "Seconds":
                time.time()
                - epoch_start,
        })

        print(
            f"{name:6s} "
            f"epoch={epoch:03d}/{r['train_epochs']} | "
            f"train={train_loss:.6f} | "
            f"val={val_for_selection:.6f} | "
            f"best={best_val:.6f}@{best_epoch} | "
            f"lr={lr:.3e} | "
            f"wait={wait}/{r['patience']}"
        )

        pd.DataFrame(
            history
        ).to_csv(
            history_path(
                name
            ),
            index=False,
        )

        if wait >= r[
            "patience"
        ]:
            print(
                f"Early stopping at epoch {epoch}."
            )
            break

    if best_state is None:
        raise RuntimeError(
            f"{name}: no best state selected."
        )

    model.load_state_dict(
        best_state
    )

    model.eval()

    ckpt = {
        "Dataset":
            name,
        "SeqLen":
            SEQ_LEN,
        "PredLen":
            PRED_LEN,
        "Seed":
            SEED,
        "Recipe":
            r,
        "OfficialRepoCommit":
            commit,
        "BestEpoch":
            best_epoch,
        "BestValMSE":
            best_val,
        "StateDict":
            best_state,
    }

    torch.save(
        ckpt,
        ckpt_path,
    )

    return (
        model,
        ckpt,
        loaders,
    )


## 10. Run ETTh1 and ETTm1 H=96 reproduction

In [11]:

results = []

for name in DATASETS:
    print(
        "\n"
        + "="*130
    )

    print(
        f"OFFICIAL PATCHTST REPRODUCTION | "
        f"{name} | L=336 | H=96"
    )

    print(
        "="*130
    )

    model, ckpt, loaders = train_or_load(
        name
    )

    official_style = evaluate_loader(
        model,
        loaders[
            "official_test"
        ],
    )

    full_stride1 = evaluate_loader(
        model,
        loaders[
            "full_test"
        ],
    )

    results.append({
        "Dataset":
            name,
        "SeqLen":
            SEQ_LEN,
        "PredLen":
            PRED_LEN,
        "BestEpoch":
            ckpt[
                "BestEpoch"
            ],
        "BestValMSE":
            ckpt[
                "BestValMSE"
            ],
        "OfficialStyle_MSE":
            official_style[
                "MSE"
            ],
        "OfficialStyle_MAE":
            official_style[
                "MAE"
            ],
        "OfficialStyle_BatchAverageMSE":
            official_style[
                "BatchAverageMSE"
            ],
        "OfficialStyle_Batches":
            official_style[
                "Batches"
            ],
        "FullStride1_MSE":
            full_stride1[
                "MSE"
            ],
        "FullStride1_MAE":
            full_stride1[
                "MAE"
            ],
        "FullStride1_Batches":
            full_stride1[
                "Batches"
            ],
        "FullTestWindows":
            len(
                loaders[
                    "test_ds"
                ]
            ),
        "OfficialEvaluatedWindows":
            (
                official_style[
                    "Batches"
                ]
                * OFFICIAL_RECIPES[
                    name
                ][
                    "batch_size"
                ]
            ),
        "OfficialRepoCommit":
            commit,
    })

    display(
        pd.DataFrame([
            results[
                -1
            ]
        ])
    )

    del (
        model,
        loaders,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


result_df = pd.DataFrame(
    results
)

result_df.to_csv(
    RESULT_ROOT
    / "official_patchtst_h96_results.csv",
    index=False,
)

display(
    result_df
)



OFFICIAL PATCHTST REPRODUCTION | ETTh1 | L=336 | H=96
ETTh1  epoch=001/100 | train=0.740868 | val=1.472355 | best=1.472355@1 | lr=1.000e-04 | wait=0/100
ETTh1  epoch=002/100 | train=0.574288 | val=0.896736 | best=0.896736@2 | lr=1.000e-04 | wait=0/100
ETTh1  epoch=003/100 | train=0.457769 | val=0.774505 | best=0.774505@3 | lr=1.000e-04 | wait=0/100
ETTh1  epoch=004/100 | train=0.420713 | val=0.724204 | best=0.724204@4 | lr=9.000e-05 | wait=0/100
ETTh1  epoch=005/100 | train=0.400857 | val=0.707788 | best=0.707788@5 | lr=8.100e-05 | wait=0/100
ETTh1  epoch=006/100 | train=0.389631 | val=0.699197 | best=0.699197@6 | lr=7.290e-05 | wait=0/100
ETTh1  epoch=007/100 | train=0.382952 | val=0.695294 | best=0.695294@7 | lr=6.561e-05 | wait=0/100
ETTh1  epoch=008/100 | train=0.378108 | val=0.689702 | best=0.689702@8 | lr=5.905e-05 | wait=0/100
ETTh1  epoch=009/100 | train=0.374847 | val=0.688810 | best=0.688810@9 | lr=5.314e-05 | wait=0/100
ETTh1  epoch=010/100 | train=0.371733 | val=0.689351 |

,Dataset,SeqLen,PredLen,BestEpoch,BestValMSE,OfficialStyle_MSE,OfficialStyle_MAE,OfficialStyle_BatchAverageMSE,OfficialStyle_Batches,FullStride1_MSE,FullStride1_MAE,FullStride1_Batches,FullTestWindows,OfficialEvaluatedWindows,OfficialRepoCommit
0,ETTh1,336,96,74,0.674419,0.37489,0.399285,0.37489,21,0.378631,0.400302,22,2785,2688,204c21efe0b39603ad6e2ca640ef5896646ab1a9



OFFICIAL PATCHTST REPRODUCTION | ETTm1 | L=336 | H=96
ETTm1  epoch=001/100 | train=0.442201 | val=0.474668 | best=0.474668@1 | lr=4.148e-06 | wait=0/20
ETTm1  epoch=002/100 | train=0.344291 | val=0.424233 | best=0.424233@2 | lr=4.591e-06 | wait=0/20
ETTm1  epoch=003/100 | train=0.315691 | val=0.405605 | best=0.405605@3 | lr=5.326e-06 | wait=0/20
ETTm1  epoch=004/100 | train=0.299587 | val=0.397015 | best=0.397015@4 | lr=6.350e-06 | wait=0/20
ETTm1  epoch=005/100 | train=0.289914 | val=0.390247 | best=0.390247@5 | lr=7.654e-06 | wait=0/20
ETTm1  epoch=006/100 | train=0.283307 | val=0.388710 | best=0.388710@6 | lr=9.233e-06 | wait=0/20
ETTm1  epoch=007/100 | train=0.277691 | val=0.384558 | best=0.384558@7 | lr=1.107e-05 | wait=0/20
ETTm1  epoch=008/100 | train=0.271492 | val=0.381561 | best=0.381561@8 | lr=1.317e-05 | wait=0/20
ETTm1  epoch=009/100 | train=0.265993 | val=0.378044 | best=0.378044@9 | lr=1.550e-05 | wait=0/20
ETTm1  epoch=010/100 | train=0.262277 | val=0.381027 | best=0.3

,Dataset,SeqLen,PredLen,BestEpoch,BestValMSE,OfficialStyle_MSE,OfficialStyle_MAE,OfficialStyle_BatchAverageMSE,OfficialStyle_Batches,FullStride1_MSE,FullStride1_MAE,FullStride1_Batches,FullTestWindows,OfficialEvaluatedWindows,OfficialRepoCommit
0,ETTm1,336,96,17,0.365308,0.289871,0.343136,0.289871,89,0.28987,0.34305,90,11425,11392,204c21efe0b39603ad6e2ca640ef5896646ab1a9


,Dataset,SeqLen,PredLen,BestEpoch,BestValMSE,OfficialStyle_MSE,OfficialStyle_MAE,OfficialStyle_BatchAverageMSE,OfficialStyle_Batches,FullStride1_MSE,FullStride1_MAE,FullStride1_Batches,FullTestWindows,OfficialEvaluatedWindows,OfficialRepoCommit
0,ETTh1,336,96,74,0.674419,0.374890,0.399285,0.374890,21,0.378631,0.400302,22,2785,2688,204c21efe0b39603ad6e2ca640ef5896646ab1a9
1,ETTm1,336,96,17,0.365308,0.289871,0.343136,0.289871,89,0.289870,0.343050,90,11425,11392,204c21efe0b39603ad6e2ca640ef5896646ab1a9


## 11. Compare with our previous baselines

In [12]:

SCREENING_SUMMARY = Path(
    "/data/dataset/strong_forecaster/"
    "multidataset_crossfit_screening/"
    "summary.csv"
)

LOOKBACK_AUDIT = Path(
    "/data/dataset/strong_forecaster/"
    "patchtst_baseline_fidelity_audit/"
    "lookback_only_H96_results.csv"
)

compare = result_df[
    [
        "Dataset",
        "FullStride1_MSE",
        "FullStride1_MAE",
        "OfficialStyle_MSE",
        "OfficialStyle_MAE",
        "BestEpoch",
    ]
].copy()

if SCREENING_SUMMARY.exists():
    old = pd.read_csv(
        SCREENING_SUMMARY
    )

    old = old[
        (
            old[
                "Dataset"
            ].isin(
                DATASETS
            )
        )
        & (
            old[
                "Horizon"
            ]
            == 96
        )
    ][
        [
            "Dataset",
            "PatchTST_MSE",
            "PatchTST_MAE",
        ]
    ].rename(
        columns={
            "PatchTST_MSE":
                "Screening_L96_MSE",
            "PatchTST_MAE":
                "Screening_L96_MAE",
        }
    )

    compare = compare.merge(
        old,
        on="Dataset",
        how="left",
    )

if LOOKBACK_AUDIT.exists():
    audit = pd.read_csv(
        LOOKBACK_AUDIT
    )[
        [
            "Dataset",
            "AuditMSE",
            "AuditMAE",
        ]
    ].rename(
        columns={
            "AuditMSE":
                "TSLib_L336_MSE",
            "AuditMAE":
                "TSLib_L336_MAE",
        }
    )

    compare = compare.merge(
        audit,
        on="Dataset",
        how="left",
    )


if "Screening_L96_MSE" in compare.columns:
    compare[
        "OfficialFullGainVsScreenL96_pct"
    ] = (
        100.0
        * (
            compare[
                "Screening_L96_MSE"
            ]
            - compare[
                "FullStride1_MSE"
            ]
        )
        / compare[
            "Screening_L96_MSE"
        ]
    )

if "TSLib_L336_MSE" in compare.columns:
    compare[
        "OfficialFullGainVsTSLibL336_pct"
    ] = (
        100.0
        * (
            compare[
                "TSLib_L336_MSE"
            ]
            - compare[
                "FullStride1_MSE"
            ]
        )
        / compare[
            "TSLib_L336_MSE"
        ]
    )

display(
    compare
)

compare.to_csv(
    RESULT_ROOT
    / "official_patchtst_h96_comparison.csv",
    index=False,
)


,Dataset,FullStride1_MSE,FullStride1_MAE,OfficialStyle_MSE,OfficialStyle_MAE,BestEpoch,Screening_L96_MSE,Screening_L96_MAE,TSLib_L336_MSE,TSLib_L336_MAE,OfficialFullGainVsScreenL96_pct,OfficialFullGainVsTSLibL336_pct
0,ETTh1,0.378631,0.400302,0.374890,0.399285,74,0.383978,0.404953,0.388915,0.410979,1.392534,2.644339
1,ETTm1,0.289870,0.343050,0.289871,0.343136,17,0.336339,0.370611,0.298168,0.350699,13.816089,2.782955



# 12. Decision rules

## A. Official-style 수치가 충분히 강하고 FullStride1도 강함

이 경우 baseline 문제는 해결된 것입니다.

그 다음에는 **같은 official PatchTST checkpoint**를 direct forecaster로 사용해
retrieval augmentation을 다시 붙입니다.

중요:

\[
\boxed{
\text{retrieval query context }L_R=96
}
\]

와

\[
\boxed{
\text{direct PatchTST context }L_D=336
}
\]

를 분리할 수 있습니다.

retrieval method의 original representation을 바꾸지 않으면서 direct forecaster만 강하게 만들 수 있기
때문입니다.

---

## B. ETTh1은 크게 좋아지고 ETTm1도 0.298 수준 또는 그 이하

가장 기대되는 시나리오입니다.

그 경우 기존 ETT screening 결과는 폐기하고 strong baseline에 대해 다시 측정합니다.

---

## C. Official implementation도 예상보다 약함

다음 항목을 먼저 확인합니다.

- official repository commit
- PyTorch 1.11 vs 현재 PyTorch version
- paper/repository reported seed variance
- official `drop_last=True` evaluation과 all-window evaluation의 차이
- source dataset version
- official result reproduction logs

**retrieval architecture를 수정하지 않습니다.**

---

# 왜 PatchTST를 이기기 어려운가?

PatchTST는 단순 Transformer baseline이 아닙니다.

장기 lookback을 patch로 압축하여 attention token 수를 줄이고,
channel-independent parameter sharing을 사용하며,
RevIN과 긴 context를 함께 이용합니다.

따라서 historical retrieval이 이기려면 단순히 좋은 과거를 찾는 것만으로는 부족하고,

\[
\boxed{
\text{strong parametric forecast가 놓치는 query에만}
}
\]

retrieved future가 추가 정보를 제공해야 합니다.

그래서 Electricity에서 나타난
**global scalar \(\alpha=0\)인데 adaptive gate는 개선**이라는 현상이 특히 중요합니다.

최종 목표는 PatchTST 전체를 대체하는 것이 아니라:

\[
\boxed{
\text{strong PatchTST}
+
\text{external historical memory}
}
\]

가 PatchTST 단독보다 좋아지는 것입니다.


## 13. Saved artifacts

In [13]:

print("Result root:", RESULT_ROOT)

for p in sorted(
    RESULT_ROOT.rglob("*")
):
    if p.is_file():
        print(
            p.relative_to(
                RESULT_ROOT
            )
        )


Result root: /data/dataset/strong_forecaster/official_patchtst_h96_reproduction
artifacts/official_repo_commit.txt
checkpoints/ETTh1_L336_H96_official_recipe_seed2021.pt
checkpoints/ETTm1_L336_H96_official_recipe_seed2021.pt
histories/ETTh1_L336_H96_official_recipe_seed2021.csv
histories/ETTm1_L336_H96_official_recipe_seed2021.csv
official_patchtst_h96_comparison.csv
official_patchtst_h96_results.csv
